# Day 11 – App Recommendation, Hybrid Content-Based Filtering & Personalized Discovery Engine

**Project**: Google Play Store Ecosystem Analysis & NLP Autocomplete/Autocorrect Analytics  
**Milestone**: Day 11 (App Recommendation, Semantic Embeddings, MMR Diversification & Cold-Start Recommender)  
**Focus**: Multi-modal app embedding space, real-time cosine similarity search, Carbonell & Goldstein MMR diversification, user portfolio recommendations, and developer cold-start benchmarking.

---

### Architectural Highlights:
1. **Multi-Modal Semantic Embedding Space**: Combines TF-IDF metadata semantics, numerical scales (installs, reviews, size, price, rating), and sentiment priors into a dense $L_2$-normalized vector space ($D=136$).
2. **Sub-Millisecond Nearest Neighbors**: Matrix dot-product vector search achieving p50 retrieval latency under 5ms across 9,659 applications.
3. **Maximal Marginal Relevance (MMR)**: Eliminates duplicate clones and filter bubbles by balancing query relevance against intra-list diversity.
4. **User Portfolio Centroid Profiling**: Multi-app installed preference vector aggregation for cross-category discovery.
5. **Developer Cold-Start Competitor Resolver**: Synthesizes unreleased app concept vectors and computes benchmark positioning.
6. **Rigorous Offline Evaluation**: Hit Rate@K, Precision@K, Recall@K, NDCG@K, Intra-List Diversity (ILD), and Catalog Coverage.


In [ ]:
# Setup workspace root and initialize recommendation engine
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add workspace root to Python path
WORKSPACE_ROOT = Path("..").resolve()
if str(WORKSPACE_ROOT) not in sys.path:
    sys.path.insert(0, str(WORKSPACE_ROOT))

from src.recommendation_engine import (
    PlayStoreRecommendationEngine,
    AppRecommendationItem,
    RecommendationBatchResult,
    ColdStartRecommendationResult
)

engine = PlayStoreRecommendationEngine()
print(f"Loaded catalog of {len(engine.apps_df)} apps.")
print(f"Embedding matrix shape: {engine.embedding_matrix.shape}")


## 1. Multi-Modal Semantic Embedding Space & L2 Normalization

Verify unit length normalization and inspect top categories across the 2D Truncated SVD projection.


In [ ]:
# Inspect L2-norm properties of embedding vectors
norms = np.linalg.norm(engine.embedding_matrix, axis=1)
print(f"Mean vector L2 norm: {np.mean(norms):.6f} (Min: {np.min(norms):.6f}, Max: {np.max(norms):.6f})")

# Pairwise similarity of sample app with itself
sample_app = 'Duolingo: Learn Languages Free'
idx = engine.find_app_index(sample_app)
self_sim = np.dot(engine.embedding_matrix[idx], engine.embedding_matrix[idx])
print(f"Self-similarity of '{sample_app}': {self_sim:.6f}")


## 2. Content-Based Cosine Similarity & Nearest-Neighbor Search

Query top recommendations across representative flagship applications.


In [ ]:
test_apps = [
    'Duolingo: Learn Languages Free',
    'Subway Surfers',
    'Spotify Music',
    'Adobe Photoshop Express:Photo Editor Collage Maker'
]

for test_app in test_apps:
    res = engine.get_similar_apps(test_app, top_k=3, diversity_weight=1.0)
    print(f"\n[Seed App]: {res.query_target} (Retrieval Latency: {res.inference_latency_ms:.2f}ms)")
    for r in res.recommendations:
        print(f"  #{r.rank}: {r.app_name} [{r.category}] - Sim: {r.similarity_score:.4f} | Installs: {r.installs:,} | Rating: {r.rating}")


## 3. Carbonell & Goldstein Maximal Marginal Relevance (MMR) Diversification

Evaluate the trade-off between pure greedy relevance ($\lambda=1.0$) and high catalog diversity ($\lambda=0.4$).


In [ ]:
query_app = 'Subway Surfers'
lambdas = [1.0, 0.7, 0.4]

print(f"MMR Diversification Analysis for '{query_app}':")
for lam in lambdas:
    res = engine.get_similar_apps(query_app, top_k=5, diversity_weight=lam)
    app_names = [r.app_name for r in res.recommendations]
    print(f"\n  Lambda = {lam:.1f} | Intra-List Diversity (ILD): {res.intra_list_diversity:.4f}")
    for r in res.recommendations:
        print(f"    #{r.rank} {r.app_name} [{r.category}] (Sim: {r.similarity_score:.4f})")


## 4. Personalized User Portfolio Recommendations & Centroid Synthesis

Given a user's multi-app portfolio, synthesize a personalized preference vector and find cross-category discoveries.


In [ ]:
user_portfolio = ['Spotify Music', 'Duolingo: Learn Languages Free', 'Subway Surfers']
portfolio_res = engine.recommend_for_user_portfolio(user_portfolio, top_k=5, diversity_weight=0.7)

print(f"User Portfolio: {user_portfolio}")
print(f"Personalized Recommendations (ILD: {portfolio_res.intra_list_diversity:.4f}, Latency: {portfolio_res.inference_latency_ms:.2f}ms):")
for r in portfolio_res.recommendations:
    print(f"  #{r.rank} {r.app_name} [{r.category}] - Match: {r.similarity_score:.4f} | Rating: {r.rating} | Installs: {r.installs:,}")


## 5. Cold-Start Developer Benchmarking & Competitor Positioning

Simulate an unreleased app concept to identify incumbent competitors and strategic advice.


In [ ]:
cold_res = engine.recommend_cold_start(
    category='FINANCE',
    genres='Finance',
    size_mb=18.5,
    price=0.0,
    content_rating='Everyone',
    top_k=5
)

print(f"Cold-Start Category: {cold_res.candidate_category} (Competition: {cold_res.category_competition_intensity})")
print(f"Category Mean Rating: {cold_res.category_mean_rating:.2f} | Median Installs: {cold_res.category_median_installs:,}")
print("Strategic Advice:")
for advice in cold_res.strategic_advice:
    print(f"  - {advice}")
print("\nTop Benchmark Incumbents:")
for r in cold_res.top_competitors:
    print(f"  #{r.rank} {r.app_name} - Match: {r.similarity_score:.4f} | Installs: {r.installs:,} | Rating: {r.rating}")


## 6. Offline Quantitative Evaluation Benchmarks & Publication Figures

Generate evaluation reports and export Figures 22 & 23.


In [ ]:
# Generate figures and reports
fig22, fig23 = engine.generate_figures()
reports = engine.generate_reports()

print(f"Figure 22: {fig22}")
print(f"Figure 23: {fig23}")
print("Reports Generated:")
for k, v in reports.items():
    print(f"  - {k}: {v}")

# Display benchmarks table
bench_df = pd.read_csv(reports['benchmarks'])
bench_df
